<a href="https://colab.research.google.com/github/GianJoshua/CC19_GIAN_PHASE_2/blob/main/CC19_DATA_MINIG_GIAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
#IMPORT STUFF
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from scipy.stats import randint

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from google.colab import files



In [27]:
# Paste the RAW url here
url = 'https://raw.githubusercontent.com/GianJoshua/CC19_GIAN_PHASE_2/refs/heads/main/synthetic_disaster_events_2025.csv'
df = pd.read_csv(url)

df.head()

,event_id,disaster_type,location,latitude,longitude,date,severity_level,affected_population,estimated_economic_loss_usd,response_time_hours,aid_provided,infrastructure_damage_index,is_major_disaster
0,1,Wildfire,Chile,-34.681672,-71.819529,2025-08-27,8,31104,2768213.39,5.12,Yes,0.59,1
1,2,Hurricane,India,22.128569,78.023951,2023-05-29,5,29340,5996226.87,44.43,No,0.26,0
2,3,Volcanic Eruption,Italy,42.316058,11.031447,2023-01-15,7,34804,9222541.48,49.30,No,0.94,1
3,4,Drought,Chile,-33.436253,-69.984615,2024-02-08,8,31191,1827703.09,65.56,Yes,0.94,1
4,5,Volcanic Eruption,Turkey,39.400977,37.006822,2023-12-23,8,46284,13435921.49,60.96,No,0.92,1


In [30]:
# APPLYING MIN-MAX SCALING TO NUMERICAL FEATUERS
# Update numeric list (excluding the target column)
exclude_from_scale = ['is_major_disaster']
scale_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude_from_scale]

# Initialize MinMaxScaler (Default range is 0 to 1)
scaler = MinMaxScaler()

# Apply the scaling
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [24]:
# Label encoding for binary variables
if 'aid_provided' in df.columns:
    le = LabelEncoder()
    df['aid_provided'] = le.fit_transform(df['aid_provided'].astype(str))

# Label encoding for categorical variables (One-Hot Encoding)
cols_to_ohe = ['disaster_type', 'location']
cols_to_ohe = [c for c in cols_to_ohe if c in df.columns]
df = pd.get_dummies(df, columns=cols_to_ohe, drop_first=True, dtype=int)

In [25]:
# Feature Engineering: Date processing
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['date'] = df['date'].fillna(df['date'].mode()[0])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.dayofweek
    df.drop(columns=['date'], inplace=True)

print("============================== TRANSFORMED DATASET PREVIEW ==============================")
print(df.head(10))

============================== TRANSFORMED DATASET PREVIEW ==============================
   event_id  latitude  longitude  severity_level  affected_population  \
0   0.00000  0.092223   0.122217        0.777778             0.413909   
1   0.00005  0.712560   0.726336        0.444444             0.390435   
2   0.00010  0.932997   0.456244        0.666667             0.463146   
3   0.00015  0.105822   0.129614        0.777778             0.415066   
4   0.00020  0.901166   0.560968        0.777778             0.615913   
5   0.00025  0.443890   0.866923        0.888889             0.660785   
6   0.00030  0.689048   0.726257        0.666667             0.445979   
7   0.00035  0.876481   0.975094        0.666667             0.461935   
8   0.00040  0.919158   0.549549        0.333333             0.106977   
9   0.00045  0.843559   0.024041        0.333333             0.276471   

   estimated_economic_loss_usd  response_time_hours  aid_provided  \
0                     0.126582       